# Add SpatialData

- Convert the Cohort of FOVs and labels to a single `SpatialData` object. 
- Save the `SpatialData` object to `LaminDB`.


## Notebook Preferences

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format="retina"

## Importing Libraries

In [2]:
from upath import UPath
import buckaroo  # noqa: F401
import pandas as pd
import natsort as ns
import lamindb as ln
from nbl._util import DaskLocalCluster
import nbl
import spatialdata as sd

Buckaroo has been enabled as the default DataFrame viewer.  To return to default dataframe visualization use `from buckaroo import disable; disable()`
💡 connected lamindb: srivarra/nbl_assets


In [3]:
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("future.infer_string", False)
pd.set_option("mode.copy_on_write", True)

In [4]:
ln.settings.transform.stem_uid = "g4NYQa7pUrPk"
ln.settings.transform.version = "1"
run = ln.track()
run.transform

💡 notebook imports: buckaroo==0.6.12 lamindb==0.74.3 natsort==8.4.0 nbl==0.0.1 pandas==2.2.2 spatialdata==0.2.2.dev16+g6b76876 universal_pathlib==0.2.2
💡 loaded: Transform(uid='g4NYQa7pUrPk5zKv', version='1', name='Add SpatialData', key='01 - Add SpatialData', type='notebook', created_by_id=1, updated_at='2024-07-30 22:35:31 UTC')
💡 loaded: Run(uid='bXgK7yQhk67w39My6kAW', started_at='2024-08-01 19:23:16 UTC', is_consecutive=True, transform_id=2, created_by_id=1)


Transform(uid='g4NYQa7pUrPk5zKv', version='1', name='Add SpatialData', key='01 - Add SpatialData', type='notebook', created_by_id=1, updated_at='2024-07-30 22:35:31 UTC')

In [5]:
ln.setup.settings.instance

Current instance: srivarra/nbl_assets
- owner: srivarra
- name: nbl_assets
- storage root: /Users/srivarra/Davis Lab/neuroblastoma/nbl/data/db
- storage region: None
- db: postgresql://srivarra:***@***:5555/nbl_db
- schema: {'bionty'}
- git_repo: None

In [6]:
cluster = DaskLocalCluster(n_workers=10, threads_per_worker=1)
cluster(open_dashboard=True)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 10
Total threads: 10,Total memory: 64.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:55561,Workers: 10
Dashboard: http://127.0.0.1:8787/status,Total threads: 10
Started: Just now,Total memory: 64.00 GiB
Comm: tcp://127.0.0.1:55585,Total threads: 1
Dashboard: http://127.0.0.1:55604/status,Memory: 6.40 GiB
Nanny: tcp://127.0.0.1:55565,


## Convert FOVs to SpatialData

### Set up Paths

In [7]:
original_data_path = UPath("../../../data/raw/original_data/nbl_cohort")
fov_dir: UPath = original_data_path / "images"
label_dir: UPath = original_data_path / "segmentation" / "labels"

In [8]:
hu_data_path: UPath = ln.settings.storage.root / "Hu.zarr"
nbl_data_path: UPath = ln.settings.storage.root / "nbl.zarr"

### Convert Control Cohort to SpatialData - Hu Data

#### Convert Cohort

In [16]:
nbl.io.convert_cohort(fov_dir=fov_dir, label_dir=label_dir, filter_fovs=r"Hu-*", file_path=hu_data_path)

  0%|          | 0/5 [00:00<?, ?it/s]

INFO     The Zarr backing store has been changed from None the new file path: /Users/srivarra/Davis                
         Lab/neuroblastoma/nbl/data/db/Hu.zarr                                                                     


In [17]:
hu_sdata = sd.read_zarr(store=hu_data_path)

#### Aggregate Images by Labels and Compute Region Properties

In [18]:
nbl.tl.aggregate_images_by_labels(sdata=hu_sdata, label_type="whole_cell", table_name="whole_cell")

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/models/models.py:970: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/models/models.py:970: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWar

In [12]:
properties = [
    "label",
    "centroid",
    "area",
    "perimeter",
    "axis_major_length",
    "axis_minor_length",
    "eccentricity",
    "orientation",
]

In [15]:
nbl.tl.regionprops(sdata=hu_sdata, label_type="whole_cell", table_name="whole_cell", properties=properties)

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/_core/_elements.py:116: UserWarning: Key `whole_cell` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


### Convert Sample Cohort to SpatialData - NBL Data

#### Convert Cohort

In [ ]:
nbl.io.convert_cohort(fov_dir=fov_dir, filter_fovs=r"NBL-\d+-R\d+C\d+", label_dir=label_dir, file_path=nbl_data_path)

In [ ]:
nbl_sdata = sd.read_zarr(store=nbl_data_path)

#### Aggregate Images by Labels and Compute Region Properties

In [ ]:
nbl.tl.aggregate_images_by_labels(sdata=nbl_sdata, label_type="whole_cell", table_name="whole_cell")

In [ ]:
properties = [
    "label",
    "centroid",
    "area",
    "perimeter",
    "axis_major_length",
    "axis_minor_length",
    "eccentricity",
    "orientation",
]

In [ ]:
nbl.tl.regionprops(sdata=nbl_sdata, label="whole_cell", table_name="whole_cell", properties=properties)

## Add Pixie Clusters to NBL SpatialData

In [ ]:
nbl_sdata.tables["whole_cell"].obs_names

In [ ]:
pixie_clusters_path = (
    original_data_path / "segmentation" / "cell_table" / "cell_table_size_normalized_cell_labels_noCD117.csv"
)

In [ ]:
pixie_clusters_df = pd.read_csv(pixie_clusters_path).astype({"label": int, "cell_meta_cluster": "category"})

In [ ]:
all_fovs = ns.natsorted(nbl_sdata.coordinate_systems)

In [ ]:
def get_pixie_clusters(df, fovs: str, suffix="whole_cell") -> pd.DataFrame:
    """Gets pixie clusters from the two clustering csv files.

    Parameters
    ----------
    df : pd.DataFrame
        The Pixie cluster DataFrame
    fov : str
        The FOV to subset on

    Returns
    -------
    pd.DataFrame
        A dataframe containing the two clusters and a column for the segmentation label.
    """
    out_df = []
    for fov in fovs:
        fov_rncm = fov.split("-")[-1].split("_")[0]
        no_cd117_pixie: pd.DataFrame = df[df["fov"].str.split("_").map(lambda x: x[-1]) == fov_rncm]
        result_df = (
            no_cd117_pixie.assign(region=f"{fov}_{suffix}")
            .rename(columns={"label": "instance_id", "cell_meta_cluster": "pixie_cluster"})
            .astype(dtype={"instance_id": int, "region": "category", "pixie_cluster": "category"})
            .filter(items=["instance_id", "region", "pixie_cluster"])
        )
        out_df.append(result_df)

    return pd.concat(out_df)

In [ ]:
pixie_df = pixie_clusters_df.pipe(func=get_pixie_clusters, fovs=all_fovs, suffix="whole_cell")

In [ ]:
nbl_sdata.tables["whole_cell"].obs = nbl_sdata.tables["whole_cell"].obs.merge(
    right=pixie_df, right_on=["instance_id", "region"], left_on=["instance_id", "region"]
)

## Add Clinical Information

### Load Clinical Data from LaminDB

In [ ]:
clinical_data: pd.DataFrame = ln.Artifact.filter(key__contains="clinical_data").one().load()

In [ ]:
clinical_data["region"] = clinical_data["fov"].map(lambda f: f"{f}_whole_cell").astype("category")

In [ ]:
cols_to_drop = ["Clinical presentation", "treatment btw biopsies", "fov"]

In [ ]:
filtered_clincial_data = clinical_data.drop(columns=cols_to_drop)

In [ ]:
nbl_sdata.tables["whole_cell"].obs = nbl_sdata.tables["whole_cell"].obs.merge(right=filtered_clincial_data, on="region")

### `Arcsinh` Transform the NBL Whole Cell Table

In [ ]:
nbl_sdata.tables["whole_cell"]

In [ ]:
nbl.pp.arcsinh_transform(sdata=nbl_sdata, table_names="whole_cell", replace_X=True, write=True)

In [ ]:
hu_artifact = ln.Artifact(data=hu_data_path, type="dataset", key="Hu.zarr", description="Control Tissue")

hu_artifact.save(upload=True)

In [ ]:
nbl_artifact = ln.Artifact(data=nbl_data_path, type="dataset", key="nbl.zarr", description="NBL Tissue Samples")

nbl_artifact.save(upload=True)

In [ ]:
ln.finish()